# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YoyuQre/flyrank_assgn_1/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

I chose the **Random Forest Classifier** for this lane because the task is a **binary classification** problem: predicting whether a content page is likely to show a downward search-performance trend. Random Forest is well suited to this problem because it can model complex relationships between historical search metrics, engagement signals, and content characteristics without assuming a linear relationship. It is also robust to noisy features and provides feature importance scores, making it useful for both prediction and interpretation. The model will be compared against the rule-based baseline developed in ML-07 to determine whether machine learning provides better prioritization.


In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df =pd.read_csv('/content/content_refresh_anonymized.csv')

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

### Split Design

I will use a **grouped train-test split based on `client_id`**. This ensures that all pages belonging to the same client are placed entirely in either the training set or the test set, preventing information from the same client from appearing in both.

This is an honest evaluation for the content refresh prediction task because the model is tested on **unseen clients**, rather than on additional pages from clients it has already learned. A random row-wise split could produce overly optimistic results by allowing the model to memorize client-specific patterns. Grouping by client better reflects how the model would perform when applied to new clients in practice.


In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Copy data
data = df.copy()

# Target
y = (data["trend_direction"] == "down").astype(int)

# Remove leakage columns
drop_cols = [
    "content_id",
    "client_id",
    "trend_direction",
    "trend_pct"
]

X = data.drop(columns=drop_cols)

# Missing values
num_cols = X.select_dtypes(include=["number"]).columns
cat_cols = X.select_dtypes(include=["object"]).columns

X[num_cols] = X[num_cols].fillna(0)
X[cat_cols] = X[cat_cols].fillna("Unknown")

# One-hot encoding
X = pd.get_dummies(X, columns=cat_cols, drop_first=True)

print(X.shape)

groups = df["client_id"]

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42
)

train_idx, test_idx = next(gss.split(X, y, groups=groups))

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

print("Train:", X_train.shape)
print("Test :", X_test.shape)

(30000, 66)
Train: (23837, 66)
Test : (6163, 66)


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.ensemble import RandomForestClassifier

# Train model
rf = RandomForestClassifier(
    n_estimators=200,
    random_state=42
)

rf.fit(X_train, y_train)

# Predictions
y_pred = rf.predict(X_test)
y_prob = rf.predict_proba(X_test)[:, 1]

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

rf_results = {
    "Accuracy": accuracy_score(y_test, y_pred),
    "Precision": precision_score(y_test, y_pred),
    "Recall": recall_score(y_test, y_pred),
    "F1 Score": f1_score(y_test, y_pred),
    "ROC-AUC": roc_auc_score(y_test, y_prob)
}

print(rf_results)

baseline_pred = (
    (df.loc[test_idx, "days_since_last_update"] >= 90) |
    (df.loc[test_idx, "ctr"] < df["ctr"].median()) |
    (df.loc[test_idx, "avg_position"] > 20) |
    (df.loc[test_idx, "impressions_90d"] < df["impressions_90d"].median())
).astype(int)

baseline_results = {
    "Accuracy": accuracy_score(y_test, baseline_pred),
    "Precision": precision_score(y_test, baseline_pred),
    "Recall": recall_score(y_test, baseline_pred),
    "F1 Score": f1_score(y_test, baseline_pred)
}

comparison = pd.DataFrame({
    "Baseline Rule": baseline_results,
    "Random Forest": {
        k: rf_results[k]
        for k in baseline_results.keys()
    }
}).T

comparison

{'Accuracy': 0.8142138568878793, 'Precision': 0.8049300060864273, 'Recall': 0.8399491902191172, 'F1 Score': 0.822066822066822, 'ROC-AUC': np.float64(0.9054267867765606)}


,Accuracy,Precision,Recall,F1 Score
Baseline Rule,0.526692,0.524608,0.785329,0.629022
Random Forest,0.814214,0.804930,0.839949,0.822067


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

The Random Forest model performed substantially better than the rule-based baseline, indicating that combining multiple signals is more effective than relying on a few fixed thresholds. However, the model can still make mistakes. Some declining pages may not show strong warning signals in the available features, leading to false negatives, while pages with unusual traffic patterns or seasonal fluctuations may be incorrectly flagged as declining, leading to false positives.

The model appears to rely on a combination of historical search performance, content freshness, engagement, and search visibility rather than any single feature. This suggests that content decline is influenced by multiple interacting factors, which explains why the Random Forest outperformed the simpler rule-based approach.


In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd

feature_importance = (
    pd.DataFrame({
        "Feature": X.columns,
        "Importance": rf.feature_importances_
    })
    .sort_values("Importance", ascending=False)
)

feature_importance.head(10)

,Feature,Importance
18,impressions_prev_30d,0.169745
15,impressions_last_30d,0.133393
5,impressions_90d,0.065951
25,avg_position,0.054366
13,days_with_impressions,0.050795
21,content_age_days,0.038262
3,word_count,0.029122
4,char_count,0.026602
17,sessions_last_30d,0.026361
24,ctr,0.024985


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.